In [17]:
import os

# Use .get() to safely check the key without throwing a KeyError
api_key = os.environ.get("OPENAI_API_KEY")

if not api_key or api_key in ["", "YOUR_API_KEY_HERE"]:
    raise Exception("API KEY MISSING OR INVALID. Please set your actual OpenAI API key.")
else:
    print("All good! Valid key found.")


from agents import Agent, Runner, function_tool
import pydantic
# Tell Pydantic not to fail when fields are missing during validation
pydantic.BaseModel.model_config['extra'] = 'ignore'

agent = Agent(
    name="MyAgent",
    model="gpt-5.4-mini",
    instructions="You are a helpful assistant.",
)


All good! Valid key found.


In [13]:
# Runner.run is a classmethod — you don't instantiate Runner.
result = await Runner.run(agent, "Hello, Where did the Minnesota Vikings play?", max_turns=10)

print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

Last Agent: MyAgent
----
The Minnesota Vikings played at **Metropolitan Stadium** in **Bloomington, Minnesota** from **1961 to 1981**, and later at the **Metrodome** in **Minneapolis** from **1982 to 2013**.


In [14]:
from tavily import TavilyClient

if os.environ["TAVILY_API_KEY"] is None:
  raise Exception("TAVILY_API_KEY MISSING")
else:
  print("all good with search")

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

all good with search


In [15]:
@function_tool
def tavily_search(query: str) -> str:
    """
    Perform a web search using Tavily and return a summarized result.
    """
    response = tavily_client.search(query,search_depth='advanced',max_results='5')
    results = response.get("results", [])
    return results or "No results found."

In [16]:
agent = Agent(
    name="Web Research Agent",
    model="gpt-5.4-mini",
    instructions="Use tavily_search when you need up-to-date info.",
    tools=[tavily_search],
)

result = await Runner.run(agent, "I'm traveling from JFK to LAX on July 17, 2026. I would like a hotel for under $300 for that night.  Options?", max_turns=10)
print(f"Last Agent: {result.last_agent.name}")
print("----")
print(result.final_output)

Last Agent: Web Research Agent
----
Yes — for July 17, 2026, you should be able to find options under $300 near LAX.

Some commonly budget-friendly areas/hotels to check:
- Sonesta Select Los Angeles LAX El Segundo
- LAX Stadium Inn (Inglewood/El Segundo)
- Los Angeles Adventurer All Suite Hotel at LAX
- Hilton Garden Inn LAX / nearby deals
- DoubleTree by Hilton LAX - El Segundo
- Hyatt Place LAX / nearby promotions

Best bets for staying under budget:
- El Segundo / Inglewood near LAX
- Search “near LAX” rather than downtown
- Consider refundable rates and airport shuttle hotels

If you want, I can help narrow it to:
1. closest to LAX,
2. nicest under $300, or
3. best value with shuttle.
